In [ ]:
# RFW manual annotation notebook
# Copy each cell into a Jupyter notebook cell, or run this file in a notebook environment.
# Requirements: ipywidgets, pillow, pandas
# Install (if needed):
# !pip install ipywidgets pillow pandas

import os
import random
from pathlib import Path
from io import BytesIO
from PIL import Image
import pandas as pd
import IPython.display as disp
import ipywidgets as widgets
import json
import time

# -----------------------------
# USER CONFIG
# -----------------------------
DATA_DIR = Path('../../data/RFW/data')  # root folder containing demographic subfolders
DEMOGRAPHICS = ['African', 'Asian', 'Caucasian', 'Indian']
SAMPLE_SIZE = 1000
OUTPUT_CSV = Path('../../results/annotations/human_annotations_1k.csv')
STATE_FILE = Path('../../results/annotations/human_annotations_1k.state.json')
AUTOSAVE_EVERY = 5  # autosave after every N changes (set to 1 for immediate save)

# Attribute schema
ATTR_SCHEMA = {
    'gender': ['male', 'female'],
    'age': ['young', 'middle-aged', 'senior'],
    'skin_color': ['light', 'medium', 'dark'],
    'ancestry': ['asian', 'south_asian', 'black', 'latino/hispanic', 'middle_eastern', 'white', 'indigenous'],
    'hair_color': ['black', 'brown', 'red', 'blonde', 'gray', 'other'],
    'bangs': ['yes', 'no'],
    'bald': ['yes', 'no'],
    'beard': ['no', 'mustache', 'stubble', 'full'],
    'glasses': ['no', 'regular', 'sun'],
    'headwear': ['no', 'beanie', 'cap', 'hat', 'headband', 'hijab', 'helmet', 'turban']
}

# -----------------------------
# Helper: Collect image paths
# -----------------------------

def collect_image_paths(data_dir: Path, demographics: list):
    paths = []
    for d in demographics:
        dem_dir = data_dir / d
        if not dem_dir.exists():
            # skip missing demographic folder
            continue
        # assuming structure data/{demographic}/{id}/{id}_{number}.jpg
        for id_folder in dem_dir.iterdir():
            if not id_folder.is_dir():
                continue
            for f in id_folder.iterdir():
                if f.is_file() and f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    paths.append((d, str(f)))
    return paths

all_paths = collect_image_paths(DATA_DIR, DEMOGRAPHICS)
print(f'Found {len(all_paths)} images across demographics (requested demographics: {DEMOGRAPHICS})')

if len(all_paths) == 0:
    raise SystemExit('No images found. Please check DATA_DIR and folder structure.')

# deterministic sampling (set seed if desired)
random.seed(42)
if len(all_paths) <= SAMPLE_SIZE:
    sampled = all_paths
else:
    sampled = random.sample(all_paths, SAMPLE_SIZE)

# Build DataFrame
records = []
for dem, path in sampled:
    p = Path(path)
    # try to extract id from parent folder name
    person_id = p.parent.name
    filename = p.name
    records.append({
        'image_path': path,
        'demographic': dem,
        'person_id': person_id,
        'filename': filename,
        # annotation fields start as empty (or you can set default values)
        **{k: '' for k in ATTR_SCHEMA.keys()}
    })

df = pd.DataFrame.from_records(records)
print('Sampled dataset size:', len(df))

# If previous CSV exists, load and merge so you can continue
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)
    # # merge by image_path -- prefer existing annotations when present
    # df = df.merge(existing, on=['image_path', 'demographic', 'person_id', 'filename'], how='left', suffixes=('', '_old'))
    # # for each attribute, if value present in _old use it
    # for attr in ATTR_SCHEMA.keys():
    #     if attr + '_old' in df.columns:
    #         df[attr] = df[attr].fillna(df[attr + '_old'])
    #         df = df.drop(columns=[attr + '_old'])
    print(f'Loaded existing annotations from {OUTPUT_CSV}, continuing where you left off.')

# Load persistent state (current index) if exists
start_index = 0
if STATE_FILE.exists():
    try:
        with open(STATE_FILE, 'r') as f:
            state_json = json.load(f)
            saved_idx = state_json.get('current_index')
            saved_time = state_json.get('timestamp')
            if isinstance(saved_idx, int) and 0 <= saved_idx < len(df):
                start_index = saved_idx
                print(f'Resuming from saved index {start_index+1} (saved at {time.ctime(saved_time)})')
    except Exception as e:
        print('Could not load state file:', e)

# -----------------------------
# UI components (use ToggleButtons for speed)
# -----------------------------

# Image display widget
img_widget = widgets.Image(format='jpeg')

# Create interactive widgets for each attribute using ToggleButtons
attr_widgets = {}
for attr, options in ATTR_SCHEMA.items():
    # include an empty option for 'unlabeled'
    opts = [''] + options
    # For long option lists, allow wrapping
    w = widgets.ToggleButtons(options=opts, description=attr + ':', layout=widgets.Layout(width='100%'))
    attr_widgets[attr] = w

# Navigation and control widgets
index_label = widgets.HTML('Image 0 / 0')
prev_btn = widgets.Button(description='<< Prev', layout=widgets.Layout(width='100px'))
next_btn = widgets.Button(description='Next >>', layout=widgets.Layout(width='100px'))
save_btn = widgets.Button(description='Save CSV', button_style='success')
jump_text = widgets.IntText(value=start_index+1, description='Jump to:', layout=widgets.Layout(width='200px'))
progress = widgets.IntProgress(value=0, min=0, max=len(df), description='Labeled:')
message = widgets.HTML('')

# internal state
state = {
    'idx': start_index,
    'dirty_count': 0
}

# -----------------------------
# Functions for display / load / save
# -----------------------------

def pil_image_to_bytes(img: Image.Image, max_size=(900, 900)):
    # Resize for display while keeping aspect ratio
    img = img.copy()
    img.thumbnail(max_size, Image.LANCZOS)
    buffer = BytesIO()
    img.save(buffer, format='JPEG')
    return buffer.getvalue()


def save_state(idx=None):
    payload = {
        'current_index': state['idx'] if idx is None else int(idx),
        'timestamp': time.time()
    }
    try:
        with open(STATE_FILE, 'w') as f:
            json.dump(payload, f)
    except Exception as e:
        print('Failed to write state file:', e)


# Fix ToggleButtons state issue: ensure widget always reflects current annotation
def load_index(i: int, set_message=True):
    i = max(0, min(i, len(df) - 1))
    state['idx'] = i
    row = df.iloc[i]
    path = row['image_path']
    try:
        pil = Image.open(path).convert('RGB')
        img_widget.value = pil_image_to_bytes(pil)
    except Exception as e:
        if set_message:
            message.value = f'<span style="color: red">Error loading image: {e}</span>'
        img_widget.value = b''

    # Force all attribute widgets to match the DataFrame
    for attr in ATTR_SCHEMA.keys():
        val = row.get(attr, '')
        if pd.isna(val) or (val not in ATTR_SCHEMA[attr]):
            val = ''
        # Update widget value only if different to avoid triggering handlers unnecessarily
        if attr_widgets[attr].value != val:
            attr_widgets[attr].value = val

    index_label.value = f'<b>Image {i+1} / {len(df)}</b> — {row["demographic"]} / {row["person_id"]} / {row["filename"]}'
    labeled = df.dropna(subset=list(ATTR_SCHEMA.keys()))
    labeled = labeled[(labeled[list(ATTR_SCHEMA.keys())] != '').all(axis=1)]
    progress.value = labeled.shape[0]

    # Save current index to state file so restart resumes at same image
    save_state()


def write_current_annotation(_save=False):
    i = state['idx']
    changed = False
    for attr in ATTR_SCHEMA.keys():
        val = attr_widgets[attr].value or ''
        if df.at[i, attr] != val:
            df.at[i, attr] = val
            changed = True
    if changed:
        state['dirty_count'] += 1
    # autosave behavior
    if _save or (state['dirty_count'] >= AUTOSAVE_EVERY):
        try:
            df.to_csv(OUTPUT_CSV, index=False)
            save_state()  # persist current index
            message.value = f'<span style="color:green">Saved {OUTPUT_CSV} and state (autosave)</span>'
            state['dirty_count'] = 0
        except Exception as e:
            message.value = f'<span style="color:red">Save failed: {e}</span>'


# Button callbacks

def on_prev(b=None):
    write_current_annotation()
    load_index(state['idx'] - 1)


def on_next(b=None):
    write_current_annotation()
    load_index(state['idx'] + 1)


def on_save(b=None):
    write_current_annotation(_save=True)
    try:
        df.to_csv(OUTPUT_CSV, index=False)
        save_state()
        message.value = f'<span style="color:green">Saved {OUTPUT_CSV} and state</span>'
    except Exception as e:
        message.value = f'<span style="color:red">Final save failed: {e}</span>'


def on_jump(change):
    # jump_text is 1-based for user
    try:
        idx = int(change['new']) - 1
    except Exception:
        message.value = '<span style="color:red">Invalid jump index</span>'
        return
    if 0 <= idx < len(df):
        write_current_annotation()
        load_index(idx)
    else:
        message.value = f'<span style="color: red">Index out of range (1..{len(df)})</span>'

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)

jump_text.observe(on_jump, names='value')

# when a widget changes, update df immediately and optionally autosave
for attr, w in attr_widgets.items():
    def make_handler(widget, attr_name):
        def handler(change):
            if change['name'] == 'value':
                # set value for current index
                df.at[state['idx'], attr_name] = widget.value or ''
                # increment dirty counter and possibly autosave
                state['dirty_count'] += 1
                if state['dirty_count'] >= AUTOSAVE_EVERY:
                    write_current_annotation(_save=True)
        return handler
    w.observe(make_handler(w, attr), names='value')

# Keyboard navigation: left/right arrows move between images
try:
    display_js = disp.Javascript('''
    if (typeof window.__rfw_key_listener_added === 'undefined') {
      window.__rfw_key_listener_added = true;
      document.addEventListener('keydown', function(e) {
          if (e.key === 'ArrowLeft') { IPython.notebook.kernel.execute("__arrow_key_pressed('left')"); }
          if (e.key === 'ArrowRight') { IPython.notebook.kernel.execute("__arrow_key_pressed('right')"); }
      });
    }
    ''')
    disp.display(display_js)
except Exception:
    # If JS injection fails (some environments), it's non-fatal.
    pass

# Expose a function the JS calls
from IPython import get_ipython

def __arrow_key_pressed(direction):
    if direction == 'left':
        on_prev(None)
    elif direction == 'right':
        on_next(None)

get_ipython().user_ns['__arrow_key_pressed'] = __arrow_key_pressed

# -----------------------------
# Layout and start
# -----------------------------

# For compactness, arrange attribute widgets into a scrollable box
attrs_box = widgets.VBox([attr_widgets[a] for a in ATTR_SCHEMA.keys()], layout=widgets.Layout(overflow='auto', max_height='800px', width='40%'))

controls_row = widgets.HBox([prev_btn, next_btn, save_btn, jump_text, index_label])
left_col = widgets.VBox([img_widget, controls_row, progress, message], layout=widgets.Layout(width='50%'))
ui = widgets.HBox([left_col, attrs_box], layout=widgets.Layout(align_items='flex-start'))

print('UI ready. Use Next/Prev to navigate, fill attributes (use ToggleButtons for speed), and Save when done.')
disp.display(ui)

# load starting image (from saved index if available)
load_index(state['idx'])

# handy function to export current dataframe to CSV on demand

def export_csv(path: str = None):
    p = Path(path) if path else OUTPUT_CSV
    df.to_csv(p, index=False)
    save_state()
    print('Exported annotations to', p)

# End of notebook script


In [ ]:
df[:755]